# Bi-level pruning - CIFAR-10 (VGG16-BN / ResNet-56)

Muc dich: chung minh bi-level tai lap duoc tren benchmark chuan, de doi chieu voi so
**published** cua L1 / HRank / GAL / SSS / CORING / SPSRC ma khong phai chay lai tung baseline.

Protocol lay tu repo CORING (`main/data/cifar10.py`, `main/scripts/resnet56_cifar10/vbd.sh`):

| | Gia tri |
|---|---|
| Transform | RandomCrop(32, pad=4) + HFlip |
| Normalize | mean (.4914,.4822,.4465) std **(.2023,.1994,.2010)** |
| Dense ResNet-56 | 200 ep, lr 0.1, x0.1 @60,120,160, wd 1e-4, bs 128 |
| Dense VGG16-BN | 164 ep, lr 0.1, x0.1 @81,122 |
| Finetune `coring` | 300 ep, lr 0.01, x0.1 @150,225, **wd 5e-3** |
| Finetune `spsrc` | 80 ep, lr 0.001, x0.1 @20, wd 1e-4 |

**Thu tu chay:** Cell 1-4 setup -> Cell 5 SMOKE TEST -> Cell 6 DENSE (co GATE) -> Cell 7 PRUNE+FINETUNE.

> **GATE**: neu dense lech >0.5% so voi so published thi notebook DUNG. Prune tren mot dense
> sai moc thi toan bo bang so sanh vo nghia - phai sua training truoc, khong duoc di tiep.

**Truoc khi chay:** Settings > Accelerator > **GPU**, va Settings > **Internet ON**
(can de tai CIFAR-10 va pip install).


In [ ]:
# --- 1. Repo + branch dev ---
import os, subprocess, sys, json, time

REPO = '/kaggle/working/OnestageDetectionPunner'
BRANCH = 'dev'

if not os.path.isdir(REPO):
    # repo private -> dung token, hoac Add Data roi copy vao /kaggle/working
    # subprocess.run(f'git clone https://<TOKEN>@github.com/barone04/OnestageDetectionPunner.git {REPO}',
    #                shell=True, check=True)
    raise SystemExit(f'{REPO} chua co. Clone repo hoac Add Data truoc.')

os.chdir(REPO)
subprocess.run(f'git fetch --all -q && git checkout {BRANCH} -q && git pull -q', shell=True)
subprocess.run('git log --oneline -1', shell=True)
print('cwd:', os.getcwd())


In [ ]:
# --- 2. .env -> os.environ (WANDB_API_KEY). Bo qua neu khong dung wandb ---
ENV = '/kaggle/input/datasets/bophaninhthi/env-file1/.env'
USE_WANDB = os.path.isfile(ENV)
if USE_WANDB:
    for line in open(ENV):
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip().strip(chr(34)).strip(chr(39))
    USE_WANDB = bool(os.environ.get('WANDB_API_KEY'))
os.environ.setdefault('WANDB_PROJECT', 'bilevel_cifar10')
print('wandb:', 'ON' if USE_WANDB else 'OFF')


In [ ]:
# --- 3. Deps + kiem tra moi truong ---
subprocess.run('pip -q install thop' + (' wandb' if USE_WANDB else ''), shell=True)
import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
print('cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Bat GPU: Settings > Accelerator > GPU'


In [ ]:
# --- 4. Config ---
MODEL    = 'resnet56'    # 'resnet56' hoac 'vgg16'
VGG_HEAD = 'hrank'       # chi cho vgg16. hrank=14.98M (HRank/CORING) | single=14.72M (SPSRC)
PROTOCOL = 'coring'      # recipe finetune sau prune: 'coring' (300ep) | 'spsrc' (80ep)
RATES    = ['0.50']      # them ['0.30','0.50','0.70'] khi con quota GPU
SEED     = 0

DATA = '/kaggle/working/data'      # CIFAR-10 tu tai ve day
OUT  = '/kaggle/working/output/cifar'
DENSE_DIR = f'{OUT}/{MODEL}_dense'
os.makedirs(DATA, exist_ok=True)

# Da co dense checkpoint tu session truoc (Add Data)? Tro vao day de bo qua Cell 6.
DENSE_CKPT_EXTERNAL = None   # vd '/kaggle/input/.../model_best.pth'

WB = '--wandb' if USE_WANDB else ''
head = f'--vgg-head {VGG_HEAD}' if MODEL == 'vgg16' else ''

def run(cmd):
    """Chay lenh, in truc tiep, dung han neu that bai."""
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise SystemExit(f'That bai (exit {r.returncode}): {cmd}')

print(f'{MODEL} | head={VGG_HEAD} | protocol={PROTOCOL} | rates={RATES}')


## 5. Smoke test

Chay het truoc khi dot GPU vao full run. Kiem tra:
1. Model dung so params so voi bang published
2. Surgery cho lean model **giong het** masked model (`max|diff| < 1e-7`)
3. Ca 3 mode dense/prune/finetune chay thong voi 1 epoch

Mat ~3-5 phut. Fail o day thi dung, dung chay tiep.


In [ ]:
# --- 5a. Self-check model + surgery ---
run('python -m models.cifar')
run('python -m pruning.surgery_cifar')


In [ ]:
# --- 5b. Smoke: dense 1 epoch (--skip-gate vi 1 epoch chac chan khong dat moc) ---
SMOKE = f'{OUT}/_smoke'
run(f'python cifar.py --mode dense --model {MODEL} {head} --data-path {DATA} '
    f'--epochs 1 --skip-gate --output-dir {SMOKE}_dense')


In [ ]:
# --- 5c. Smoke: prune (2 vong x 1 ep) + surgery ---
run(f'python cifar.py --mode prune --model {MODEL} {head} --data-path {DATA} '
    f'--checkpoint {SMOKE}_dense/model_best.pth --protocol {PROTOCOL} '
    f'--target-sparsity 0.5 --prune-iters 2 --prune-finetune-epochs 1 '
    f'--output-dir {SMOKE}_p50')


In [ ]:
# --- 5d. Smoke: finetune 1 epoch tren lean model ---
run(f'python cifar.py --mode finetune --data-path {DATA} --protocol {PROTOCOL} '
    f'--lean {SMOKE}_p50/model_lean.pth --epochs 1 --output-dir {SMOKE}_ft')
print(json.load(open(f'{SMOKE}_p50/cost.json')))
print(chr(10) + 'SMOKE TEST PASS - duoc phep chay that.')


## 6. Dense baseline (co GATE)

ResNet-56 200 epoch ~ 1-1.5h tren P100/T4. VGG16-BN 164 epoch ~ 1.5-2h.

**Cell nay tu dung neu dense lech >0.5% so voi published.** Do la co y:
prune tren dense sai moc thi khong duoc trich bang cua nguoi ta.

| Model | Target (HRank/CORING) |
|---|---|
| ResNet-56 | 93.26% |
| VGG16-BN (head=hrank) | 93.96% |

Da co dense tu session truoc thi set `DENSE_CKPT_EXTERNAL` o Cell 4 va **bo qua cell nay**.


In [ ]:
# --- 6. Dense (bo qua neu da co checkpoint ngoai) ---
if DENSE_CKPT_EXTERNAL:
    DENSE_CKPT = DENSE_CKPT_EXTERNAL
    print('Dung dense co san:', DENSE_CKPT)
else:
    t0 = time.time()
    run(f'python cifar.py --mode dense --model {MODEL} {head} --data-path {DATA} '
        f'--seed {SEED} --output-dir {DENSE_DIR} {WB}')
    DENSE_CKPT = f'{DENSE_DIR}/model_best.pth'
    print(f'Dense xong sau {(time.time()-t0)/3600:.2f}h')

assert os.path.isfile(DENSE_CKPT), DENSE_CKPT
print('top-1 dense:', torch.load(DENSE_CKPT, map_location='cpu', weights_only=False)['top1'])


## 7. Prune + finetune

Moi rate: bi-level prune (5 vong) -> surgery -> finetune theo `PROTOCOL`.
Voi `protocol='coring'` (300 epoch) moi rate ton ~1.5-2h - canh quota 12h/session cua Kaggle.
Chay 1 rate/session la an toan nhat.


In [ ]:
# --- 7. Bi-level prune + finetune cho tung rate ---
results = {}
for R in RATES:
    p_dir = f'{OUT}/{MODEL}_p{R}'
    f_dir = f'{OUT}/{MODEL}_p{R}_ft'
    print(chr(10) + '=' * 64 + chr(10) + f' RATE {R}' + chr(10) + '=' * 64, flush=True)

    run(f'python cifar.py --mode prune --model {MODEL} {head} --data-path {DATA} '
        f'--checkpoint {DENSE_CKPT} --protocol {PROTOCOL} --seed {SEED} '
        f'--target-sparsity {R} --prune-iters 5 --prune-finetune-epochs 3 '
        f'--output-dir {p_dir}')

    run(f'python cifar.py --mode finetune --data-path {DATA} --protocol {PROTOCOL} '
        f'--seed {SEED} --lean {p_dir}/model_lean.pth --output-dir {f_dir} {WB}')

    cost = json.load(open(f'{p_dir}/cost.json'))
    ck = torch.load(f'{f_dir}/model_best.pth', map_location='cpu', weights_only=False)
    results[R] = dict(cost, top1=ck['top1'])
    print(f"RATE {R} -> top1={ck['top1']:.2f}%  {cost}")


In [ ]:
# --- 8. Bang tong ket (dan thang vao .tex duoc) ---
hdr = ('rate', 'top1', 'params M', '-params%', 'MACs M', '-MACs%')
print('{:>6} | {:>7} | {:>9} | {:>9} | {:>9} | {:>8}'.format(*hdr))
print('-' * 66)
for R, r in results.items():
    print('{:>6} | {:>6.2f}% | {:>9.3f} | {:>8.2f}% | {:>9.2f} | {:>7.2f}%'.format(
        R, r['top1'], r['params_M'], r['params_red_pct'], r['macs_M'], r['macs_red_pct']))
path = f'{OUT}/summary_{MODEL}_{PROTOCOL}.json'
json.dump(results, open(path, 'w'), indent=2)
print(chr(10) + 'Luu: ' + path)
